# Interactive Climate Bias-Correction Pipeline
This notebook clones the repository, installs dependencies, and lets you upload a shapefile, select parameters, and run the full QDM bias-correction pipeline.

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Install dependencies
!git clone https://github.com/awan-geospatial1/climate-downscaling.git /content/climate-downscaling
!pip install -q -r /content/climate-downscaling/requirements.txt --upgrade


# Add to Python path
import sys
sys.path.insert(0, '/content/climate-downscaling')

# Import modules
import os, tempfile, zipfile, pandas as pd, ipywidgets as widgets
from IPython.display import display, clear_output
from main import run_pipeline
# --- Define widgets ---
shapefile_upload = widgets.FileUpload(accept='.zip,.shp,.gpkg,.geojson', multiple=False, description='Upload shapefile')
drive_path_input = widgets.Text(value='', placeholder='Or enter Drive path', description='Drive path:', layout=widgets.Layout(width='80%'))
gee_project = widgets.Text(value='', placeholder='GCP project ID', description='GEE Project ID:')
all_gcms = ['ACCESS-CM2','ACCESS-ESM1-5','AWI-CM-1-1-MR','BCC-CSM2-MR','CAMS-CSM1-0','CAS-ESM2-0','CESM2','CESM2-WACCM','CMCC-CM2-SR5','CNRM-CM6-1','CNRM-CM6-1-HR','CNRM-ESM2-1','CanESM5','EC-Earth3','EC-Earth3-Veg','EC-Earth3-Veg-LR','FGOALS-f3-L','FGOALS-g3','FIO-ESM-2-0','GFDL-CM4','GFDL-ESM4','GISS-E2-1-G','INM-CM4-8','INM-CM5-0','IPSL-CM6A-LR','KACE-1-0-G','MIROC-ES2L','MIROC6','MPI-ESM1-2-HR','MPI-ESM1-2-LR','MRI-ESM2-0','NESM3','NorESM2-LM','NorESM2-MM']
gcm_select = widgets.SelectMultiple(options=all_gcms, value=['EC-Earth3','CNRM-CM6-1','GFDL-ESM4','MPI-ESM1-2-LR','GISS-E2-1-G'], description='GCMs')
scenario_select = widgets.SelectMultiple(options=['ssp245','ssp370','ssp585'], value=['ssp245','ssp585'], description='Scenarios')
baseline_start = widgets.DatePicker(description='Baseline start', value=pd.to_datetime('1990-01-01'))
baseline_end = widgets.DatePicker(description='Baseline end', value=pd.to_datetime('2014-12-31'))
hist_start = widgets.DatePicker(description='Historical plot start', value=pd.to_datetime('1990-01-01'))
future_intervals = widgets.Textarea(value='2026-01-01,2050-12-31,Short,2026-2050\n2051-01-01,2075-12-31,Mid,2051-2075\n2076-01-01,2100-12-31,Long,2076-2100', description='Future intervals', layout=widgets.Layout(width='80%', height='100px'))
month_options = [f'{i:02d}' for i in range(1,13)]
wet_months = widgets.SelectMultiple(options=month_options, value=['05','06','07','08','09','10'], description='Wet months')
temp_thresholds = widgets.Text(value='30.0', description='T thresholds (°C, comma)')
precip_thresholds = widgets.Text(value='20.0,25.0', description='P thresholds (mm, comma)')
return_periods = widgets.Text(value='100', description='Return periods (years, comma)')
gev_bootstrap = widgets.IntText(value=1000, description='GEV bootstrap')
output_dir = widgets.Text(value='/content/drive/MyDrive/climate_output', description='Output folder:')
buffer_km = widgets.FloatText(value=25.0, description='Buffer (km)')
nquantiles = widgets.IntText(value=50, description='QDM quantiles')
wet_thresh = widgets.FloatText(value=0.1, description='Wet-day threshold (mm)')
run_button = widgets.Button(description='▶️ Run Pipeline', button_style='success')
output_area = widgets.Output()

# Display
display(shapefile_upload, drive_path_input, gee_project, gcm_select, scenario_select,
        baseline_start, baseline_end, hist_start, future_intervals, wet_months,
        temp_thresholds, precip_thresholds, return_periods, gev_bootstrap,
        output_dir, buffer_km, nquantiles, wet_thresh, run_button, output_area)

def get_shapefile_path():
    if shapefile_upload.value:
        for fname, info in shapefile_upload.value.items():
            content = info['content']
            if fname.endswith('.zip'):
                temp_dir = tempfile.mkdtemp()
                zip_path = os.path.join(temp_dir, fname)
                with open(zip_path, 'wb') as f: f.write(content)
                with zipfile.ZipFile(zip_path, 'r') as zf: zf.extractall(temp_dir)
                shp_files = [f for f in os.listdir(temp_dir) if f.endswith('.shp')]
                if shp_files:
                    return os.path.join(temp_dir, shp_files[0])
                else:
                    raise ValueError("No .shp found in zip")
            else:
                temp_dir = tempfile.mkdtemp()
                path = os.path.join(temp_dir, fname)
                with open(path, 'wb') as f: f.write(content)
                return path
    elif drive_path_input.value.strip():
        return drive_path_input.value.strip()
    else:
        raise ValueError("No shapefile provided")

def on_run(b):
    with output_area:
        clear_output()
        try:
            shp_path = get_shapefile_path()
            params = {
                'shapefile_path': shp_path,
                'buffer_km': buffer_km.value,
                'gee_project_id': gee_project.value.strip(),
                'models': list(gcm_select.value),
                'scenarios': list(scenario_select.value),
                'baseline_start': baseline_start.value.strftime('%Y-%m-%d'),
                'baseline_end': baseline_end.value.strftime('%Y-%m-%d'),
                'hist_start': hist_start.value.strftime('%Y-%m-%d'),
                'future_intervals': [tuple(p.strip() for p in line.split(',')) for line in future_intervals.value.strip().split('\n') if line.strip()],
                'wet_months': [int(m) for m in wet_months.value],
                'dry_months': [m for m in range(1,13) if m not in [int(m) for m in wet_months.value]],
                'temp_thresholds': [float(x) for x in temp_thresholds.value.split(',') if x.strip()],
                'precip_thresholds': [float(x) for x in precip_thresholds.value.split(',') if x.strip()],
                'return_periods': [int(x) for x in return_periods.value.split(',') if x.strip()],
                'gev_n_bootstrap': gev_bootstrap.value,
                'nquantiles': nquantiles.value,
                'wet_thresh': wet_thresh.value,
                'output_dir': output_dir.value.strip(),
            }
            run_pipeline(params)
        except Exception as e:
            print(f"❌ ERROR: {e}")
            import traceback; traceback.print_exc()

run_button.on_click(on_run)
print("✅ Interactive widgets loaded. Fill in the parameters and click 'Run Pipeline'.")